# Fine-tune SAM ViT-B สำหรับภาพอัลตราซาวด์
เลือก Runtime → Change runtime type → GPU แล้วรันทีละเซลล์ ฝึก mask decoder เท่านั้น SAM ประมาณขอบเขต ไม่ได้วินิจฉัยมะเร็ง อ่าน sam_training/README.md ก่อนใช้ผลคะแนน

In [ ]:
import subprocess, sys
from pathlib import Path
repo = Path("/content/My-Little-Project")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/aunyarit2005-hash/My-Little-Project.git", str(repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "sam_training/requirements.txt")], check=True)
import torch
print("GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "เลือก GPU runtime ก่อนเริ่มฝึก"

## ข้อมูล
อัปโหลด dataset เข้ารันไทม์เอง หรือรันเซลล์ Drive ด้านล่างเพื่อเข้าถึงโฟลเดอร์ส่วนตัว แล้วตั้ง DATA/GROUPS เป็น path จริง การใช้ Colab จะประมวลผลข้อมูลบนบริการ Google

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATA = "/content/drive/MyDrive/breast_dataset"
GROUPS = "/content/drive/MyDrive/groups.csv"  # CSV: image,group
ALLOW_IMAGE_SPLIT = False  # True เฉพาะงานทดลองที่ไม่มี patient IDs
OUT = Path("/content/drive/MyDrive/sam_runs/run_01")
MANIFEST = Path("/content/drive/MyDrive/sam_runs/manifest_01.json")
BASE = Path("/content/sam_vit_b_01ec64.pth")
EPOCHS = 20
assert Path(DATA).is_dir(), "แก้ DATA ให้ตรงกับโฟลเดอร์ภาพและ mask"

## ดาวน์โหลด pretrained SAM จาก Meta
ไฟล์ base นี้ต้องเก็บไว้ใช้คู่กับ decoder ที่ฝึกเสร็จ

In [ ]:
import urllib.request
if not BASE.exists():
    tmp = BASE.with_suffix(".download")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", tmp)
    tmp.rename(BASE)

In [ ]:
def run(script, *args):
    subprocess.run([sys.executable, str(repo / "sam_training" / script), *map(str, args)], check=True)
if not MANIFEST.exists():
    split_args = ["--allow-image-split"] if ALLOW_IMAGE_SPLIT else ["--groups", GROUPS]
    run("data.py", "--data", DATA, "--out", MANIFEST, *split_args)
else:
    print("ใช้ manifest เดิม:", MANIFEST)

In [ ]:
run("train.py", "--data", DATA, "--manifest", MANIFEST, "--base", BASE, "--out", OUT, "--epochs", EPOCHS, "--device", "cuda")

## Held-out test
รันหลังเลือกโมเดลเสร็จ คะแนนนี้ใช้ BB จาก ground-truth mask ไม่ใช่ผล YOLO→SAM ทั้งระบบ

In [ ]:
run("train.py", "--data", DATA, "--manifest", OUT / "manifest.json", "--base", BASE, "--evaluate", OUT / "best_decoder.pt", "--device", "cuda")

## ทดลองภาพใหม่
แก้ IMAGE และ BOX เป็นภาพจริงกับกรอบ xyxy ในพิกเซล หรือใช้ --yolo ตาม README

In [ ]:
IMAGE = "/content/drive/MyDrive/example.png"
BOX = [50, 60, 180, 200]
PRED = OUT / "example_prediction"
run("predict.py", "--image", IMAGE, "--base", BASE, "--decoder", OUT / "best_decoder.pt", "--box", *BOX, "--out", PRED)
from IPython.display import display, Image
display(Image(filename=str(PRED / "overlay.png")))